In [ ]:
#| default_exp ai

# AI

> Vertex AI Gemini (`google-genai`), Vector Search, Vertex AI Search.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import os
try:
    from google import genai
    from google.genai import types as genai_types
except ImportError:
    pass

try:
    from google.cloud import aiplatform
    from google.cloud.aiplatform import MatchingEngineIndex, MatchingEngineIndexEndpoint
    from google.cloud import discoveryengine_v1 as discoveryengine
except ImportError:
    pass

def _genai_client(auth):
    """Return a configured google-genai Client for Agent Platform (Vertex AI).

    Uses ``GOOGLE_CLOUD_LOCATION`` (default ``'global'``) rather than
    ``auth.region`` because the Agent Platform global endpoint provides
    automatic routing to the nearest available region — this is the recommended
    default per the google/skills ``gemini-api`` skill.  Override with
    ``GOOGLE_CLOUD_LOCATION=us-central1`` if a specific region is required
    (e.g. for data residency constraints).
    """
    return genai.Client(
        vertexai=True,
        project=auth.project,
        location=os.environ.get('GOOGLE_CLOUD_LOCATION', 'global'),
        credentials=auth.credentials,
    )

#: Curated fallback list of Gemini models known to be available on Vertex AI
#: at time of writing.  ``list_models`` will try a live SDK call first and
#: fall back to this list on any error (offline tests, missing API perms, …).
_FALLBACK_MODELS = [
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-2.0-flash',
    'gemini-1.5-pro',
    'gemini-1.5-flash',
    'text-embedding-005',
    'text-embedding-004',
]


def list_models(auth) -> list:
    """List Gemini model IDs available on Agent Platform (Vertex AI).

    Attempts a live ``client.models.list()`` call; falls back to a curated
    list of models known to exist if the SDK call fails (e.g. offline tests
    or missing API enablement).
    """
    try:
        client = _genai_client(auth)
        ids = []
        for m in client.models.list():
            mid = getattr(m, 'name', None) or getattr(m, 'model', None)
            if not mid:
                continue
            ids.append(mid.rsplit('/', 1)[-1])
        return ids or list(_FALLBACK_MODELS)
    except Exception:
        return list(_FALLBACK_MODELS)


def generate_content(
    auth,
    prompt: str,
    model: str = 'gemini-2.5-flash',
    max_tokens: int = 1024,
    safety_settings: dict = None,
    **_,
) -> str:
    """Generate a text response via the google-genai SDK on Agent Platform.

    Uses the unified `google-genai` SDK (not the deprecated `vertexai` SDK).
    Set ``GOOGLE_CLOUD_LOCATION=global`` (default) for automatic regional routing.

    Pass ``safety_settings`` as a dict mapping
    ``google.genai.types.HarmCategory`` to ``HarmBlockThreshold``, e.g.::

        from google.genai import types
        safety_settings = {
            types.HarmCategory.HARM_CATEGORY_HATE_SPEECH:
                types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        }
    """
    client = _genai_client(auth)
    config_kwargs = {'max_output_tokens': max_tokens}
    if safety_settings:
        config_kwargs['safety_settings'] = safety_settings
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=genai_types.GenerateContentConfig(**config_kwargs),
    )
    return response.text

def create_vector_search_index(
    auth,
    name: str,
    dimensions: int = 768,
    approximate_neighbors: int = 150,
    distance_measure: str = 'DOT_PRODUCT_DISTANCE',
    labels: dict = None,
    wait: bool = True,
    **_,
) -> dict:
    """Create a Vertex AI Vector Search (Matching Engine) index.

    Index creation is a long-running op (typically 20–40 minutes).  Pass
    ``wait=False`` to return immediately with a handle once the LRO has
    been submitted, and poll separately.
    """
    aiplatform.init(project=auth.project, location=auth.region,
                    credentials=auth.credentials)
    if wait:
        idx = MatchingEngineIndex.create_tree_ah_index(
            display_name=name,
            dimensions=dimensions,
            approximate_neighbors_count=approximate_neighbors,
            distance_measure_type=distance_measure,
            labels=labels or {},
        )
        return {'name': idx.resource_name, 'display_name': name, 'status': 'ready'}
    # Async path: submit the LRO and return.  ``sync=False`` is the SDK flag
    # that makes the call non-blocking; the returned object exposes
    # ``.resource_name`` once available.
    idx = MatchingEngineIndex.create_tree_ah_index(
        display_name=name,
        dimensions=dimensions,
        approximate_neighbors_count=approximate_neighbors,
        distance_measure_type=distance_measure,
        labels=labels or {},
        sync=False,
    )
    return {
        'name': getattr(idx, 'resource_name', None) or '',
        'display_name': name,
        'status': 'pending',
    }


def create_vector_search_endpoint(
    auth,
    name: str,
    public: bool = False,
    labels: dict = None,
    **_,
) -> dict:
    """Create a Vertex AI Vector Search index endpoint."""
    aiplatform.init(project=auth.project, location=auth.region,
                    credentials=auth.credentials)
    ep = MatchingEngineIndexEndpoint.create(
        display_name=name,
        public_endpoint_enabled=public,
        labels=labels or {},
    )
    return {'name': ep.resource_name, 'display_name': name}

def create_search_app(
    auth,
    name: str,
    data_store_type: str = 'GENERIC',
    **_,
) -> dict:
    """Create a Vertex AI Search data store + app for RAG pipelines."""
    client = discoveryengine.DataStoreServiceClient(
        credentials=auth.credentials
    )
    parent = f'projects/{auth.project}/locations/global/collections/default_collection'
    data_store = discoveryengine.DataStore(
        display_name=name,
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )
    op = client.create_data_store(
        parent=parent,
        data_store=data_store,
        data_store_id=name.replace(' ', '-').lower(),
    )
    result = op.result(timeout=300)
    return {'name': result.name, 'display_name': name}


def search_query(
    auth,
    app_id: str,
    query: str,
    page_size: int = 10,
) -> list:
    """Run a search query against a Vertex AI Search app."""
    client = discoveryengine.SearchServiceClient(
        credentials=auth.credentials
    )
    serving_config = (
        f'projects/{auth.project}/locations/global/collections/default_collection'
        f'/engines/{app_id}/servingConfigs/default_config'
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
    )
    response = client.search(request=req)
    return [r.document.struct_data for r in response.results]


### Tests — module exports

In [ ]:
#| hide
import gcpeasy.ai as M
for n in ['list_models', 'generate_content']:
    assert n in M.__all__